# C2 - StudentLife PRELIMINARY MODELS (step 4)

**Purpose:** train and compare two simple models on the table built in step 3, and check honestly whether they learn anything.

**Reminder - what this is and is not**
- Target = `y_stressed` (EMA answers: stressed 2/3 vs feeling good 4/5). It is a **stress** label from a public dataset, **not** a C2 capacity label.
- StudentLife is university students at one US college, not working students. Everything below is **PRELIMINARY** and for learning the pipeline. It is not validation of C2.
- Inputs are phone-sensing features only (your C2 design). Participant ID is never a feature.

## Models
1. **Majority-class baseline** - always guesses the more common class. Any real model must beat this.
2. **Logistic Regression** (baseline model, simple and explainable). Missing values are filled with the median and features are scaled **inside** the pipeline, so nothing leaks from test to train.
3. **Random Forest** (candidate). `min_samples_leaf=5` limits overfitting (my choice, not tuned).

## How we evaluate (no accuracy-only judging)
- **Accuracy** - share of correct answers. Misleading alone.
- **Precision** - when the model says "stressed", how often it is right.
- **Recall** - of the really stressed answers, how many the model found.
- **F1** - one number balancing precision and recall.
- **ROC-AUC** - how well the model ranks stressed above not-stressed (0.5 = guessing, 1.0 = perfect).
- **Confusion matrix** - the four counts: true/false positives and negatives.
- Two evaluations: (A) the fixed train/test people from step 3, (B) **grouped cross-validation**: the data is cut into 5 folds where whole people are held out each time. B is more trustworthy because the fixed test set has only 10 people.
- **Shuffled-label control:** we also train on randomly shuffled labels. If that scores clearly above 0.5, something is leaking. It should be about 0.5.

## Cell 1 - Connect Drive and load the step 3 files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
table = pd.read_csv(OUT_DIR / 'studentlife_model_table.csv')
with open(OUT_DIR / 'studentlife_split.json') as fh:
    split = json.load(fh)

FEATURES = split['features']
X_all, y_all, groups_all = table[FEATURES], table['y_stressed'], table['uid']
train_mask = table['uid'].isin(split['train_uids'])
test_mask = table['uid'].isin(split['test_uids'])
assert not (set(split['train_uids']) & set(split['test_uids'])), 'LEAK: participant in both sets'
assert 'uid' not in FEATURES, 'participant id must not be a feature'

print('rows:', len(table), '| participants:', table['uid'].nunique(), '| features:', len(FEATURES))
print('train rows:', int(train_mask.sum()), '| test rows:', int(test_mask.sum()))
print('share stressed (all):', round(y_all.mean(), 3))

## Cell 2 - Define the three models
Each model is a **pipeline**: the preparation steps (fill missing values, scale) are learned only from the training part, then applied to the test part.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

def make_models():
    return {
        'Majority-class baseline': Pipeline([('clf', DummyClassifier(strategy='most_frequent'))]),
        'Logistic Regression': Pipeline([('imp', SimpleImputer(strategy='median')),
                                         ('scale', StandardScaler()),
                                         ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
        'Random Forest': Pipeline([('imp', SimpleImputer(strategy='median')),
                                   ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                                  random_state=42, n_jobs=-1))]),
    }
print(list(make_models()))

## Cell 3 - Evaluation A: the fixed train/test people
Train on the 37 training people, test on the 10 unseen people. Positive class = stressed (1).
Remember: only about 150 test rows, so these numbers are noisy.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

Xtr, ytr = X_all[train_mask], y_all[train_mask]
Xte, yte = X_all[test_mask], y_all[test_mask]

fixed_rows, fixed_models, fixed_cm = [], {}, {}
for name, model in make_models().items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    fixed_models[name] = model
    fixed_cm[name] = confusion_matrix(yte, pred, labels=[0, 1])
    fixed_rows.append({'model': name,
                       'accuracy': accuracy_score(yte, pred),
                       'precision': precision_score(yte, pred, zero_division=0),
                       'recall': recall_score(yte, pred, zero_division=0),
                       'f1': f1_score(yte, pred, zero_division=0),
                       'roc_auc': roc_auc_score(yte, proba)})

fixed_results = pd.DataFrame(fixed_rows).set_index('model').round(3)
display(fixed_results)
for name, cm in fixed_cm.items():
    print(f'\nConfusion matrix - {name}   (rows = truth, columns = prediction; order: 0 feeling good, 1 stressed)')
    print(cm)

## Cell 4 - Evaluation B: grouped cross-validation on ALL people (the number to trust more)
5 folds. Each fold holds out whole people, and stressed/not-stressed shares are kept similar (`StratifiedGroupKFold`).
We report the **mean and spread (std) across folds**. A big spread means the result is unstable.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy',
           'precision': make_scorer(precision_score, zero_division=0),
           'recall': make_scorer(recall_score, zero_division=0),
           'f1': make_scorer(f1_score, zero_division=0),
           'roc_auc': 'roc_auc'}

cv_rows = []
for name, model in make_models().items():
    res = cross_validate(model, X_all, y_all, groups=groups_all, cv=cv, scoring=scoring)
    row = {'model': name}
    for m in scoring:
        row[f'{m}_mean'] = res[f'test_{m}'].mean()
        row[f'{m}_std'] = res[f'test_{m}'].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index('model').round(3)
display(cv_results[[c for c in cv_results.columns if c.endswith('_mean')]])
print('spread across folds (std):')
display(cv_results[[c for c in cv_results.columns if c.endswith('_std')]])

## Cell 5 - Confusion matrices from the cross-validation (all 1,159 answers, each predicted while its person was held out)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cv_cm = {}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, model) in zip(axes, make_models().items()):
    pred = cross_val_predict(model, X_all, y_all, groups=groups_all, cv=cv)
    cm = confusion_matrix(y_all, pred, labels=[0, 1])
    cv_cm[name] = cm
    ConfusionMatrixDisplay(cm, display_labels=['good (0)', 'stressed (1)']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig(OUT_DIR / 'studentlife_confusion_matrices.png', dpi=120)
plt.show()
for name, cm in cv_cm.items():
    print(name, '->', cm.tolist())

## Cell 6 - Control checks (is it really learning, or leaking?)
1. **Shuffled labels:** same pipeline, but the labels are randomly shuffled. Expected: roc_auc about 0.5.
2. **Beat the baseline?** A real model must clearly beat the majority-class baseline AND the shuffled control.

In [ ]:
rng = np.random.RandomState(0)
y_shuffled = pd.Series(rng.permutation(y_all.values), index=y_all.index)

ctrl_rows = []
for name in ('Logistic Regression', 'Random Forest'):
    res = cross_validate(make_models()[name], X_all, y_shuffled, groups=groups_all, cv=cv,
                         scoring={'accuracy': 'accuracy', 'roc_auc': 'roc_auc'})
    ctrl_rows.append({'model': name + ' (SHUFFLED labels)',
                      'accuracy_mean': res['test_accuracy'].mean(), 'roc_auc_mean': res['test_roc_auc'].mean()})
display(pd.DataFrame(ctrl_rows).set_index('model').round(3))

print('\nReal models (grouped CV) vs controls:')
display(cv_results.loc[:, ['accuracy_mean', 'f1_mean', 'roc_auc_mean']])

## Cell 7 - Which features mattered? (exploratory only)
From the models trained on the training people. **This shows what the model used, not what causes stress.**

In [ ]:
lr = fixed_models['Logistic Regression'].named_steps['clf']
rf = fixed_models['Random Forest'].named_steps['clf']

lr_imp = pd.Series(lr.coef_[0], index=FEATURES).sort_values(key=np.abs, ascending=False)
rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Logistic Regression - top 8 by |coefficient| (features are scaled, positive = more stressed):')
display(lr_imp.head(8).round(3).to_frame('coef'))
print('Random Forest - top 8 importances:')
display(rf_imp.head(8).round(3).to_frame('importance'))

## Cell 8 - Choose the preliminary model and export it
Rule (decided in advance): choose the model with the higher **mean F1 in grouped cross-validation**. Then refit it on all people and save it with `joblib`.
The saved file is a **prototype only**, so it can be loaded later by the backend. It is not a validated C2 model.

In [ ]:
import joblib
from sklearn.base import clone

candidates = cv_results.loc[['Logistic Regression', 'Random Forest'], 'f1_mean']
best_name = candidates.idxmax()
print('selected by mean grouped-CV F1:', best_name, '| F1 =', candidates[best_name])

final_model = clone(make_models()[best_name]).fit(X_all, y_all)
payload = {'model': final_model, 'features': FEATURES, 'target': 'y_stressed',
           'status': 'PRELIMINARY prototype trained on StudentLife stress labels; NOT a validated C2 model',
           'selected_by': 'mean grouped-CV F1'}
joblib.dump(payload, OUT_DIR / 'c2_prelim_studentlife_model.joblib')

summary = pd.concat({'fixed_split': fixed_results,
                     'grouped_cv': cv_results[[c for c in cv_results.columns if c.endswith('_mean')]]}, axis=0)
summary.to_csv(OUT_DIR / 'studentlife_model_results.csv')
print('saved model + results table in', OUT_DIR)

## Cell 9 - What to send back to Claude
Send the output of Cells 3, 4, 5, 6, 7 and 8 (the tables and numbers are enough).